# 02 · Transformação Camada Silver (CineData Analytics)

O objetivo desta etapa é aplicar regras de qualidade, limpeza e padronização sobre os dados brutos da camada Bronze. 

**Nesta camada, aplicamos:**
- Deduplicação de registros.
- Padronização de nomes de colunas e tipagem.
- Tratamento de nulos e strings inconsistentes.
- Regras de negócio específicas (ex: conversão de moedas, tradução de status).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cinedata_analytics"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Schema {silver_schema} garantido. Iniciando transformações...")

Schema cinedata_analytics.silver garantido. Iniciando transformações...


## 0. Funções auxiliares (reutilizadas nas tabelas abaixo)

- `normalizar_numero`: converte texto "sujo" em texto numérico limpo (ou `NULL`). Resolve, de forma única, os problemas de
  símbolo de moeda, separador de milhar e separador decimal (`.` vs `,`) que aparecem em orçamento, receita, popularidade, notas e votos.
  **Só aceita o valor se, depois da limpeza, ele for de fato um número.** Isso evita que um texto deslocado (Column Shift) como
  `"Action 2"` vire o número `2` por engano.
- `para_inteiro`: usada em contagens (votos). Só aceita números inteiros e não negativos.

In [0]:
def normalizar_numero(coluna, separador_unico_e_milhar=False):
    """
    Devolve uma coluna STRING contendo um número no padrão '1234.56' (ponto decimal, sem milhar) ou NULL.

    separador_unico_e_milhar:
        Define como interpretar um separador ÚNICO seguido de exatamente 3 dígitos (ex.: '1.234' ou '1,234').
        - True  -> milhar (usar em valores inteiros por natureza: orçamento, receita, contagem de votos)
        - False -> decimal (usar em popularidade e notas, que são fracionárias)
    """
    # Remove símbolos/códigos de moeda e espaços.
    s = F.regexp_replace(F.trim(coluna.cast("string")), r"(?i)(us\$|r\$|usd|brl|\$|€|£)", "")
    s = F.regexp_replace(s, r"\s", "")

    qtd_pontos = F.length(s) - F.length(F.regexp_replace(s, r"\.", ""))
    qtd_virgulas = F.length(s) - F.length(F.regexp_replace(s, ",", ""))

    normalizado = (
        # Notação científica (ex.: 1.5E8) já está pronta para o cast
        F.when(s.rlike(r"^-?\d+(\.\d+)?[eE][+-]?\d+$"), s)
        # Tem '.' e ',' -> o ÚLTIMO separador é o decimal
        .when(s.contains(".") & s.contains(","),
              F.when(s.rlike(r",[^.]*$"),  # vírgula depois do último ponto -> formato BR (1.234,56)
                     F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", "."))
               .otherwise(F.regexp_replace(s, ",", "")))  # formato US (1,234.56)
        # Só vírgula
        .when(s.contains(","),
              F.when((qtd_virgulas > 1) | (F.lit(separador_unico_e_milhar) & s.rlike(r"^-?\d{1,3},\d{3}$")),
                     F.regexp_replace(s, ",", "")) # milhar
               .otherwise(F.regexp_replace(s, ",", ".")))  # decimal
        # Só ponto
        .when(s.contains("."),
              F.when((qtd_pontos > 1) | (F.lit(separador_unico_e_milhar) & s.rlike(r"^-?\d{1,3}\.\d{3}$")),
                     F.regexp_replace(s, r"\.", "")) # milhar
               .otherwise(s)) # decimal
        .otherwise(s)
    )

    # Validação final: se não parece um número, vira NULL
    return F.when(normalizado.rlike(r"^-?\d+(\.\d+)?([eE][+-]?\d+)?$"), normalizado)


def para_inteiro(coluna):
    """Contagem segura: aceita só inteiros >= 0. Fracionário (ex.: nota que 'escorregou' para a coluna de votos) vira NULL."""
    n = normalizar_numero(coluna, separador_unico_e_milhar=True).try_cast("double")
    return F.when((n % 1 == 0) & (n >= 0), n).try_cast("int")


def normalizar_id(nome_coluna="id"):
    """Chave natural sempre STRING e sem espaços (o PDF define id_filme como STRING)."""
    return F.trim(F.col(nome_coluna).cast("string"))

## 1. tb_info_filmes

**Regras aplicadas:**
- Deduplicação: Mantém apenas o registro mais recente baseado no `ingestion_datetime`.
- Mapeamento: Renomeia as colunas para português.
- Limpeza do status: remove tudo que não for letra (hífens, espaços extras, símbolos) e padroniza a caixa **antes** de traduzir.
  Assim `Post-Production`, `post production` e `POST_PRODUCTION` viram a mesma chave (`postproduction`). Valores não mapeados viram "Não Informado".
- Datas: testa vários padrões (com dia/mês de 1 ou 2 dígitos). Só vira `NULL` se nenhum padrão funcionar. Criação da coluna derivada `ano_lancamento`.
- Tipagem: `duracao_minutos` INT; `id_filme` STRING.

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info").withColumn("id", normalizar_id("id"))

# Deduplicação baseada na data de ingestão mais recente
window_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_info_dedup = (
    df_bronze_info
    .withColumn("rn", F.row_number().over(window_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Padrões de data aceitos. 'd' e 'M' (1 letra) aceitam dia/mês com 1 ou 2 dígitos.
formatos_data = ["yyyy-M-d", "d/M/yyyy", "M/d/yyyy", "yyyy/M/d", "d-M-yyyy", "M-d-yyyy",
                 "yyyyMMdd", "d MMM yyyy", "MMM d, yyyy", "MMMM d, yyyy"]

df_silver_info = (
    df_info_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")

    # Duração: cast seguro para INT; zero ou negativo não é duração válida -> NULL
    .withColumn("duracao_temp", normalizar_numero(F.col("runtime")).try_cast("double").try_cast("int"))
    .withColumn("duracao_minutos", F.when(F.col("duracao_temp") > 0, F.col("duracao_temp")))

    # Limpeza e Tradução do Status: só letras, minúsculas -> chave comparável
    .withColumn("status_chave", F.lower(F.regexp_replace(F.col("status").cast("string"), r"[^a-zA-Z]", "")))
    .withColumn("status_filme",
        F.when(F.col("status_chave") == "released", "Lançado")
        .when(F.col("status_chave") == "postproduction", "Pós-Produção")
        .when(F.col("status_chave") == "inproduction", "Em Produção")
        .when(F.col("status_chave") == "planned", "Planejado")
        .when(F.col("status_chave") == "rumored", "Rumores")
        .when(F.col("status_chave").isin("canceled", "cancelled"), "Cancelado")
        .otherwise("Não Informado")
    )

    # Data multi-formato: remove eventual parte de hora (2020-01-05 00:00:00 / 2020-01-05T00:00:00) e testa cada padrão
    .withColumn("data_txt", F.regexp_replace(F.trim(F.col("release_date").cast("string")), r"[T ]\d{2}:\d{2}.*$", ""))
    .withColumn("data_lancamento", F.coalesce(*[F.try_to_date("data_txt", fmt) for fmt in formatos_data]))
    .withColumn("ano_lancamento", F.year("data_lancamento"))

    .select("id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
            "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao")
)

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

print("[OK] silver.tb_info_filmes processada com sucesso.")
display(spark.table(f"{silver_schema}.tb_info_filmes").limit(5))

[OK] silver.tb_info_filmes processada com sucesso.


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null
1000094,Le clan,Le clan,2023-01-18,2023,90,fr,Lançado,"Fred, Achille, Max and Belette form a good-for-nothing gang of crooks. Dismally failing their last raid, they decide to get back in the game, kidnapping Sophie Marceau.",null


## 2. tb_financeiro_filmes

**Regras aplicadas:**
- Deduplicação por id do filme.
- Limpeza: remoção de símbolo de moeda, milhar e textos de ausência ("Unknown", "Não Informado"...) via `normalizar_numero` antes do cast para `decimal(18,2)`.
- Validação: orçamento e receita zerados ou negativos são forçados a NULL.
- Conversão cambial: usa a cotação mais recente extraída na Bronze (`tb_cotacao_dolar`).
- Lucro: `receita - orçamento`, tratando ausente como 0 **desde que ao menos um dos dois exista** (o PDF pede que valores ausentes não invalidem a conta). Se ambos ausentes, o lucro é NULL.
- Margem (%): `lucro / receita * 100`, protegida contra divisão por zero.

In [0]:
# Resgate da cotação mais recente (desempate pela ingestão mais recente)
linha_cotacao = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc())
    .select("cotacaoCompra").first()
)
if linha_cotacao is None:
    raise ValueError("bronze.tb_cotacao_dolar está vazia. Execute a ingestão da API (notebook 01) antes da Silver.")
cotacao_atual = float(linha_cotacao[0])

df_bronze_fin = spark.table(f"{bronze_schema}.tb_movies_financials").withColumn("id", normalizar_id("id"))

# Deduplicação
window_dedup_fin = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_fin_dedup = df_bronze_fin.withColumn("rn", F.row_number().over(window_dedup_fin)).filter(F.col("rn") == 1).drop("rn")


def limpar_moeda(nome_coluna):
    """Texto -> decimal(18,2). Ausência textual, zero e negativo viram NULL."""
    valor = normalizar_numero(F.col(nome_coluna), separador_unico_e_milhar=True).try_cast("decimal(18,2)")
    return F.when(valor > 0, valor)


def diferenca_segura(receita, orcamento):
    """receita - orçamento com ausente tratado como 0; NULL somente se os dois forem ausentes."""
    return (
        F.when(receita.isNull() & orcamento.isNull(), F.lit(None))
        .otherwise(F.coalesce(receita, F.lit(0)) - F.coalesce(orcamento, F.lit(0)))
        .cast("decimal(18,2)")
    )


df_silver_fin = (
    df_fin_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumn("orcamento_usd", limpar_moeda("budget"))
    .withColumn("receita_usd", limpar_moeda("revenue"))

    # Conversão para Reais
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.lit(cotacao_atual)).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.lit(cotacao_atual)).cast("decimal(18,2)"))

    # Lucro
    .withColumn("lucro_usd", diferenca_segura(F.col("receita_usd"), F.col("orcamento_usd")))
    .withColumn("lucro_brl", diferenca_segura(F.col("receita_brl"), F.col("orcamento_brl")))

    # Margem de lucro (%): só calcula com receita > 0 
    .withColumn("margem_lucro_pct",
        F.when(F.col("receita_usd") > 0, (F.col("lucro_usd") / F.col("receita_usd")) * 100).cast("decimal(18,2)")
    )

    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_pct")
)

df_silver_fin.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

print(f"[OK] silver.tb_financeiro_filmes gerada. Cotação utilizada: R$ {cotacao_atual:.4f}")
display(spark.table(f"{silver_schema}.tb_financeiro_filmes").limit(5))

[OK] silver.tb_financeiro_filmes gerada. Cotação utilizada: R$ 5.4000


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
1000004,null,null,null,null,null,null,null
1000005,null,null,null,null,null,null,null
1000007,null,null,null,null,null,null,null
1000011,null,null,null,null,null,null,null
1000014,null,null,null,null,null,null,null


## 3. tb_metricas_engajamento

**Regras aplicadas:**
- Deduplicação por id do filme.
- Limpeza numérica com `normalizar_numero`: trata vírgula/ponto decimal, milhar e remove textos antes do cast, sem converter lixo em número.
- Safe cast (Column Shift): textos deslocados viram `NULL`. Contagens de votos só aceitam inteiros >= 0.
- Limites de negócio: notas fora de 0–10 e popularidade negativa viram `NULL`.
- **Popularidade com "ano" (Column Shift):** valores **inteiros entre 1888 e 2100** são anos que escorregaram de outra coluna
  (a popularidade real do TMDB é fracionária, ex.: 28.482). Esses valores são invalidados (`NULL`).


In [0]:
df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics").withColumn("id", normalizar_id("id"))

# Deduplicação
window_dedup_met = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_met_dedup = df_bronze_metrics.withColumn("rn", F.row_number().over(window_dedup_met)).filter(F.col("rn") == 1).drop("rn")

# Limites de ano (primeiro filme da história: 1888) usados para detectar "ano no lugar da popularidade"
ANO_MIN, ANO_MAX = 1888, 2100

# Teto baseado em testes feitos com os valores já limpos
POPULARIDADE_MAX = 5000

pop = F.col("popularidade_temp")
parece_ano = (pop % 1 == 0) & (pop >= ANO_MIN) & (pop <= ANO_MAX)
dentro_do_teto = (pop <= F.lit(POPULARIDADE_MAX)) if POPULARIDADE_MAX is not None else F.lit(True)

df_silver_metrics = (
    df_met_dedup
    .withColumnRenamed("id", "id_filme")

    # Popularidade
    .withColumn("popularidade_temp", normalizar_numero(F.col("popularity")).try_cast("double"))
    .withColumn("popularidade", F.when((pop >= 0) & ~parece_ano & dentro_do_teto, pop))

    # Notas: 0 a 10. Fora disso vira NULL
    .withColumn("nota_tmdb_temp", normalizar_numero(F.col("vote_average")).try_cast("double"))
    .withColumn("nota_media_tmdb", F.when(F.col("nota_tmdb_temp").between(0, 10), F.col("nota_tmdb_temp")))
    .withColumn("nota_imdb_temp", normalizar_numero(F.col("averageRating")).try_cast("double"))
    .withColumn("nota_media_imdb", F.when(F.col("nota_imdb_temp").between(0, 10), F.col("nota_imdb_temp")))

    # Contagens de votos: inteiros >= 0
    .withColumn("qtd_votos_tmdb", para_inteiro(F.col("vote_count")))
    .withColumn("qtd_votos_imdb", para_inteiro(F.col("numVotes")))

    .select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
)

df_silver_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

print("[OK] silver.tb_metricas_engajamento processada com sucesso.")
display(spark.table(f"{silver_schema}.tb_metricas_engajamento").limit(5))

[OK] silver.tb_metricas_engajamento processada com sucesso.


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.132,0.0,0,6.8,27
1000005,0.6,0.0,0,null,40
1000007,0.6,0.0,0,4.7,10
1000011,1.169,0.0,0,4.8,16
1000014,0.6,0.0,0,7.2,15


## 4. tb_avaliacoes_usuarios

**Regras aplicadas:**
- Deduplicação de registros **integralmente** idênticos (filme + usuário + nota + comentário), comparando os valores **brutos**
  (antes de invalidar notas), para não eliminar avaliações distintas que só ficariam iguais depois de a nota virar `NULL`.
- Nota: conversão segura (aceita vírgula decimal) e limitada ao intervalo 0–10; fora disso vira `NULL`.
- Comentário: vazio, nulo ou só espaços vira o texto padronizado **"Sem comentário"**.

In [0]:
df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_mapped = (
    df_bronze_reviews
    .withColumn("id_filme", normalizar_id("id"))
    .withColumn("nome_usuario", F.trim(F.col("nome").cast("string")))
    .withColumn("nota_bruta", F.trim(F.col("nota").cast("string")))
    .withColumn("comentario_bruto", F.trim(F.col("comentario").cast("string")))
)

# Deduplicação sobre os valores brutos
df_dedup_rev = df_mapped.dropDuplicates(["id_filme", "nome_usuario", "nota_bruta", "comentario_bruto"])

df_silver_reviews = (
    df_dedup_rev
    .withColumn("comentario_usuario",
        F.when(F.col("comentario_bruto").isNull() | (F.col("comentario_bruto") == ""), F.lit("Sem comentário"))
        .otherwise(F.col("comentario_bruto"))
    )
    .withColumn("nota_temp", normalizar_numero(F.col("nota_bruta")).try_cast("double"))
    .withColumn("nota_usuario", F.when(F.col("nota_temp").between(0, 10), F.col("nota_temp")))
    .select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")
)

df_silver_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

print("[OK] silver.tb_avaliacoes_usuarios processada com sucesso.")
display(spark.table(f"{silver_schema}.tb_avaliacoes_usuarios").limit(5))

[OK] silver.tb_avaliacoes_usuarios processada com sucesso.


id_filme,nome_usuario,nota_usuario,comentario_usuario
797814,Matheus Cavalcanti,1.1,Péssimo em todos os sentidos.
800797,Sérgio Almeida 335,7.2,Filme interessante com boas atuações.
1086795,Rodrigo Oliveira 273,9.7,Excepcional! História envolvente e atuações impecáveis.
1054508,Tatiana Oliveira 369,7.7,Legal! Uma boa opção para o fim de semana.
776552,Letícia Lopes,8.4,Filme interessante com boas atuações.


## 5. tb_generos

**Regras aplicadas:**
- Padronização de separadores: `;` e `|` viram `,` antes da quebra.
- Separação e desmembramento: `split` + `explode`, um gênero por filme.
- Padronização de caixa (`initcap`): evita que `drama` e `Drama` virem dois gêneros diferentes na `dim_genres`.
- Filtro de qualidade (Column Shift): só passa texto formado por letras (espaço, hífen, `&` e apóstrofo permitidos) com até 30 caracteres.
  Isso remove números, datas, notas deslocadas e textos descritivos longos.
- Deduplicação: gêneros repetidos para o mesmo filme são removidos.

In [0]:
# NOTA:
# O item 5 do PDF cita "participante/empresa" e "entidades" na regra de desmembramento. 
# Trata-se de uma inconsistência no documento. 
# Para preservar a integridade do Star Schema na camada Gold e evitar mistura de domínios, 
# esta tabela (tb_generos) processa estritamente a coluna 'genres'. 
# Atores, diretores e produtoras serão isolados na dimensão tb_pessoas_empresas.

df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags").withColumn("id", normalizar_id("id"))

df_silver_generos = (
    df_bronze_credits
    .select(F.col("id").alias("id_filme"), F.col("genres").cast("string").alias("genres"))
    .withColumn("genres_norm", F.regexp_replace(F.col("genres"), "[;|]", ","))
    .withColumn("nome_genero", F.explode_outer(F.split(F.col("genres_norm"), ",")))
    .withColumn("nome_genero", F.initcap(F.regexp_replace(F.trim(F.col("nome_genero")), r"\s+", " ")))
    .filter(
        F.col("nome_genero").isNotNull() &
        F.col("nome_genero").rlike(r"^[A-Za-zÀ-ÿ][A-Za-zÀ-ÿ \-&']*$") &
        (F.length(F.col("nome_genero")) <= 30) &
        F.col("nome_genero").isin( # Whitelist criada para excluir qualquer termo que não seja um genero
            "Action", "Adventure", "Animation", "Comedy", "Crime",
            "Documentary", "Drama", "Family", "Fantasy", "History",
            "Horror", "Music", "Mystery", "Romance", "Science Fiction",
            "Tv Movie", "Thriller", "War", "Western"
        )
    )
    .select("id_filme", "nome_genero")
    .dropDuplicates()
)

df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

print("[OK] silver.tb_generos processada com sucesso.")

display(spark.table(f"{silver_schema}.tb_generos").groupBy("nome_genero").count().orderBy(F.desc("count")))

[OK] silver.tb_generos processada com sucesso.


nome_genero,count
Drama,32720
Documentary,19342
Comedy,18881
Thriller,10418
Horror,9878
Romance,7730
Action,6123
Crime,4791
Animation,4537
Tv Movie,4121


## 6. tb_pessoas_empresas

**Regras aplicadas:**
- Desmembramento e limpeza: separadores inconsistentes (`|` e `;`) viram `,`; espaços repetidos são colapsados.
- Padronização: capitalização (iniciais maiúsculas) via `initcap`.
- Filtro de qualidade (Column Shift): exige ao menos uma letra (descarta números e datas deslocados) e no máximo 100 caracteres (descarta textos descritivos).
- Consolidação: `cast`, `directors`, `writers` e `production_companies` unidos em um modelo único via `unionAll`.
- Tipagem de entidade: mapeamento estático ('Ator', 'Diretor', 'Roteirista', 'Produtora').
- Deduplicação: remoção de entidades repetidas para o mesmo filme e papel.

In [0]:
def extrair_entidades(df, nome_coluna, tipo_entidade):
    return (
        df.select(F.col("id").alias("id_filme"), F.col(nome_coluna).cast("string").alias(nome_coluna))
        .withColumn("texto_norm", F.regexp_replace(F.col(nome_coluna), "[;|]", ","))
        .withColumn("nome_pessoa", F.explode_outer(F.split(F.col("texto_norm"), ",")))
        .withColumn("nome_pessoa", F.initcap(F.regexp_replace(F.trim(F.col("nome_pessoa")), r"\s+", " ")))
        .withColumn("tipo_entidade", F.lit(tipo_entidade))
        .filter(
            F.col("nome_pessoa").isNotNull() &
            F.col("nome_pessoa").rlike(r"[A-Za-zÀ-ÿ]") &
            (F.length(F.col("nome_pessoa")) <= 100)
        )
        .select("id_filme", "nome_pessoa", "tipo_entidade")
    )

df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags").withColumn("id", normalizar_id("id"))

df_atores = extrair_entidades(df_bronze_credits, "cast", "Ator")
df_diretores = extrair_entidades(df_bronze_credits, "directors", "Diretor")
df_roteiristas = extrair_entidades(df_bronze_credits, "writers", "Roteirista")
df_produtoras = extrair_entidades(df_bronze_credits, "production_companies", "Produtora")

df_silver_pessoas_empresas = (
    df_atores
    .unionAll(df_diretores)
    .unionAll(df_roteiristas)
    .unionAll(df_produtoras)
    .dropDuplicates()
)

df_silver_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

print("[OK] silver.tb_pessoas_empresas processada com sucesso.")
display(spark.table(f"{silver_schema}.tb_pessoas_empresas").limit(10))

[OK] silver.tb_pessoas_empresas processada com sucesso.


id_filme,nome_pessoa,tipo_entidade
337339,Luke Evans,Ator
537915,Khadijha Red Thunder,Ator
527774,Dichen Lachman,Ator
396422,Stephanie Sigman,Ator
718930,Andrew Koji,Ator
635744,Udi Razzin,Ator
452507,Dylan Schmid,Ator
882598,Kal Penn,Ator
449562,Douggie Mcmeekin,Ator
339967,Nathan Ellison,Ator


## 7. tb_cotacao_dolar

**Regras aplicadas:**
- **Uma cotação por dia:** a API PTAX devolve vários boletins por dia útil e a Bronze é `append` (reexecuções duplicam linhas).
  Mantemos a cotação de horário mais recente do dia (fechamento), desempatando pela ingestão mais recente.
- Série temporal contínua: calendário completo entre a menor e a maior data.
- Forward Fill: finais de semana e feriados recebem a última cotação útil conhecida.

In [0]:
df_bronze_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# Uma linha por dia (cotação mais tardia do dia; em empate, ingestão mais recente)
window_dia = Window.partitionBy("data_origem").orderBy(F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc())

df_dolar_base = (
    df_bronze_dolar
    .withColumn("data_origem", F.col("dataHoraCotacao").try_cast("timestamp").cast("date"))
    .filter(F.col("data_origem").isNotNull())
    .withColumn("rn", F.row_number().over(window_dia))
    .filter(F.col("rn") == 1)
    .select("data_origem", "cotacaoCompra")
)

min_data = df_dolar_base.select(F.min("data_origem")).first()[0]
max_data = df_dolar_base.select(F.max("data_origem")).first()[0]

# Calendário contínuo (inclui fins de semana)
df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{min_data}'), to_date('{max_data}'), interval 1 day)) AS data_referencia
""")

# Cruza calendário x cotações reais (fins de semana/feriados ficam NULL)
df_dolar_continuo = df_calendario.join(df_dolar_base, df_calendario.data_referencia == df_dolar_base.data_origem, "left")

# Forward Fill: último valor não nulo olhando para trás
window_ffill = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_dolar = (
    df_dolar_continuo
    .withColumn("cotacao_preenchida", F.last("cotacaoCompra", ignorenulls=True).over(window_ffill))
    .select(
        F.col("data_referencia").alias("data_cotacao"),
        F.col("cotacao_preenchida").alias("valor_cotacao").cast("decimal(10,4)")
    )
)

df_silver_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

print("[OK] silver.tb_cotacao_dolar processada. Forward Fill aplicado.")
display(spark.table(f"{silver_schema}.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] silver.tb_cotacao_dolar processada. Forward Fill aplicado.


data_cotacao,valor_cotacao
2026-10-05,5.4000
2026-10-04,5.4300
2026-10-03,5.4600
2026-10-02,5.4600
2026-10-01,5.4900
2026-09-30,5.5200
2026-09-29,5.5500
2026-09-28,5.5800
